In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Данные о зачисленных абитуриентах
ENROLLMENT_DATA = {
    '2023': 686,
    '2024': 768,
    '2025': 808
}

# Информация о файлах
FILES = {
    'Профессионалитет ЗТТиЭ': {
        '2023': 'Profi-23.xlsx',
        '2024': 'Profi-24.xlsx',
        '2025': 'Profi-25.xlsx'
    },
    'ЗТТиЭ': {
        '2023': 'ZTTiE-23.xlsx',
        '2024': 'ZTTiE-24.xlsx',
        '2025': 'ZTTiE-25.xlsx'
    }
}

def load_and_aggregate_data():
    """Загружает все файлы и агрегирует данные по годам"""
    aggregated = {}
    
    for year in ['2023', '2024', '2025']:
        total_posts = 0
        total_likes = 0
        total_reposts = 0
        total_comments = 0
        total_er_values = []
        posts_count = 0
        
        for source, files in FILES.items():
            filename = files[year]
            try:
                df = pd.read_excel(filename)
                print(f"  Загружен {filename}: {len(df)} постов")
                
                # Считаем посты
                total_posts += len(df)
                
                # Суммируем лайки, репосты, комментарии
                if 'Likes' in df.columns:
                    total_likes += pd.to_numeric(df['Likes'], errors='coerce').fillna(0).sum()
                if 'Reposts' in df.columns:
                    total_reposts += pd.to_numeric(df['Reposts'], errors='coerce').fillna(0).sum()
                if 'Comments' in df.columns:
                    total_comments += pd.to_numeric(df['Comments'], errors='coerce').fillna(0).sum()
                
                # Собираем ER Post для среднего
                if 'ER Post' in df.columns:
                    er_values = pd.to_numeric(df['ER Post'], errors='coerce').dropna()
                    total_er_values.extend(er_values.values)
                    posts_count += len(er_values)
                    
            except FileNotFoundError:
                print(f"  Файл {filename} не найден")
                continue
        
        # Рассчитываем средний ER Post
        avg_er = np.mean(total_er_values) if total_er_values else 0
        
        aggregated[year] = {
            'posts': total_posts,
            'likes': total_likes,
            'reposts': total_reposts,
            'comments': total_comments,
            'avg_er': avg_er,
            'enrollment': ENROLLMENT_DATA[year]
        }
        
        print(f"\nИТОГИ {year}:")
        print(f"  Постов: {total_posts}")
        print(f"  Лайков: {total_likes:.0f}")
        print(f"  Репостов: {total_reposts:.0f}")
        print(f"  Комментариев: {total_comments:.0f}")
        print(f"  Средний ER: {avg_er:.4f}")
        print(f"  Зачислено: {ENROLLMENT_DATA[year]}")
    
    return aggregated

def analyze_correlation(data_dict, x_key, x_label, y_key='enrollment'):
    """Анализирует корреляцию между переменной X и Y (зачисление)"""
    years = sorted(data_dict.keys())
    x_values = [data_dict[year][x_key] for year in years]
    y_values = [data_dict[year][y_key] for year in years]
    
    # Рассчитываем корреляцию Пирсона
    correlation = np.corrcoef(x_values, y_values)[0, 1]
    
    # Строим регрессионную модель
    X = np.array(x_values).reshape(-1, 1)
    y = np.array(y_values)
    
    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    
    # Коэффициенты
    slope = model.coef_[0]
    intercept = model.intercept_
    
    # Прогноз на 2026 (экстраполяция на 1 шаг)
    # Предполагаем линейный рост X (используем средний прирост)
    x_2026 = x_values[-1] + (x_values[-1] - x_values[0]) / 2
    y_2026_pred = model.predict([[x_2026]])[0]
    
    return {
        'x_values': x_values,
        'y_values': y_values,
        'correlation': correlation,
        'r2': r2,
        'slope': slope,
        'intercept': intercept,
        'x_2026': x_2026,
        'y_2026_pred': y_2026_pred,
        'years': years,
        'x_label': x_label
    }

def plot_correlation(results, filename):
    """Строит графики для всех метрик"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Корреляция медиапоказателей с числом зачисленных абитуриентов', 
                 fontsize=16, fontweight='bold')
    
    metrics = [
        ('posts', 'Количество постов'),
        ('likes', 'Сумма лайков'),
        ('reposts', 'Сумма репостов'),
        ('comments', 'Сумма комментариев'),
        ('avg_er', 'Средний ER Post')
    ]
    
    colors = plt.cm.Set1(np.linspace(0, 1, len(metrics)))
    
    for idx, (metric, label) in enumerate(metrics):
        row = idx // 3
        col = idx % 3
        ax = axes[row, col]
        
        analysis = results[metric]
        
        # Точки данных
        ax.scatter(analysis['x_values'], analysis['y_values'], 
                  color=colors[idx], s=100, zorder=5)
        
        # Линия регрессии
        x_line = np.linspace(min(analysis['x_values']), max(analysis['x_values']), 100)
        y_line = analysis['slope'] * x_line + analysis['intercept']
        ax.plot(x_line, y_line, '--', color='gray', alpha=0.7, 
                label=f'R² = {analysis["r2"]:.3f}')
        
        # Подписи годов
        for i, year in enumerate(analysis['years']):
            ax.annotate(year, (analysis['x_values'][i], analysis['y_values'][i]),
                       xytext=(5, 5), textcoords="offset points", fontsize=9)
        
        ax.set_xlabel(label)
        ax.set_ylabel('Зачислено абитуриентов')
        ax.set_title(f'{label}\nКорреляция: {analysis["correlation"]:.3f}')
        ax.grid(True, alpha=0.3)
        ax.legend()
    
    # Убираем пустой подграфик
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\nГрафик сохранен: {filename}")

def plot_forecast(results, filename):
    """Строит график прогноза на 2026 год с разноцветными столбцами"""
    fig, ax = plt.subplots(figsize=(14, 7))
    
    years = ['2023', '2024', '2025', '2026 (прогноз)']
    
    # Фактические зачисления
    actual = [ENROLLMENT_DATA[y] for y in ['2023', '2024', '2025']]
    
    # Собираем прогнозы по метрикам с R² > 0.7
    forecast_data = []
    for metric, analysis in results.items():
        if analysis['r2'] > 0.7:
            forecast_data.append({
                'label': analysis['x_label'],
                'value': analysis['y_2026_pred'],
                'color': plt.cm.tab10(len(forecast_data) % 10),  # разные цвета из палитры
                'r2': analysis['r2']
            })
    
    # Если нет сильных корреляций, берем лучшую
    if not forecast_data:
        best_metric = max(results.items(), key=lambda x: x[1]['r2'])
        forecast_data.append({
            'label': best_metric[1]['x_label'],
            'value': best_metric[1]['y_2026_pred'],
            'color': 'gray',
            'r2': best_metric[1]['r2']
        })
    
    # Сортируем прогнозы по значению для наглядности
    forecast_data.sort(key=lambda x: x['value'])
    
    # Позиции для столбцов
    x_pos = np.arange(len(years))
    
    # Столбцы для фактических данных
    bars_actual = ax.bar(x_pos[:3], actual, width=0.6, 
                         label='Факт', color='skyblue', edgecolor='black', linewidth=1)
    
    # Столбцы для прогнозов (размещаем рядом друг с другом)
    n_forecasts = len(forecast_data)
    forecast_width = 0.6 / n_forecasts  # ширина каждого прогнозного столбца
    
    for i, forecast in enumerate(forecast_data):
        # Рассчитываем позицию для каждого прогнозного столбца
        offset = (i - n_forecasts/2 + 0.5) * forecast_width
        
        bar = ax.bar(x_pos[3] + offset, forecast['value'], 
                     width=forecast_width * 0.9,  # небольшой зазор между столбцами
                     label=f"{forecast['label']} (R²={forecast['r2']:.2f})", 
                     color=forecast['color'], edgecolor='black', linewidth=1, alpha=0.8)
        
        # Добавляем значение над столбцом
        ax.text(x_pos[3] + offset, forecast['value'] + 5, 
                f'{forecast["value"]:.0f}', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Добавляем значения над фактическими столбцами
    for i, val in enumerate(actual):
        ax.text(i, val + 5, str(val), ha='center', va='bottom', 
                fontsize=10, fontweight='bold')
    
    # Настройка осей и заголовков
    ax.set_xlabel('Год', fontsize=12)
    ax.set_ylabel('Количество зачисленных', fontsize=12)
    ax.set_title('Прогноз зачисления абитуриентов на 2026 год\n(по разным медиапоказателям)', 
                 fontsize=14, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(years, fontsize=11)
    
    # Добавляем легенду справа от графика
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=10)
    
    # Добавляем сетку для удобства чтения
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    
    # Устанавливаем границы Y с небольшим запасом
    max_value = max(max(actual), max(f['value'] for f in forecast_data))
    ax.set_ylim(0, max_value * 1.1)
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"График прогноза сохранен: {filename}")

def print_analysis_table(results):
    """Выводит таблицу с результатами анализа"""
    print("\n" + "="*80)
    print("РЕЗУЛЬТАТЫ КОРРЕЛЯЦИОННОГО АНАЛИЗА")
    print("="*80)
    
    data = []
    for metric, analysis in results.items():
        data.append({
            'Метрика': analysis['x_label'],
            'Корреляция': f"{analysis['correlation']:.3f}",
            'R²': f"{analysis['r2']:.3f}",
            'Сила связи': 'Сильная' if abs(analysis['correlation']) > 0.7 else 
                          ('Средняя' if abs(analysis['correlation']) > 0.5 else 'Слабая'),
            'Направление': 'Положительная' if analysis['correlation'] > 0 else 'Отрицательная',
            'Уравнение': f"y = {analysis['slope']:.2f}·x + {analysis['intercept']:.0f}",
            'Прогноз 2026': f"{analysis['y_2026_pred']:.0f}"
        })
    
    df = pd.DataFrame(data)
    print(df.to_string(index=False))
    
    # Сохраняем в Excel
    df.to_excel('корреляционный_анализ.xlsx', index=False)
    print("\nТаблица сохранена: корреляционный_анализ.xlsx")
    
    # Вывод лучшего прогноза
    best = max(results.items(), key=lambda x: x[1]['r2'])
    print(f"\nЛУЧШИЙ ПРОГНОЗ на 2026 год:")
    print(f"  По метрике: {best[1]['x_label']}")
    print(f"  R² = {best[1]['r2']:.3f}")
    print(f"  Прогноз: {best[1]['y_2026_pred']:.0f} абитуриентов")
    print(f"  Уравнение: y = {best[1]['slope']:.2f}·x + {best[1]['intercept']:.0f}")

def main():
    print("="*80)
    print("АНАЛИЗ КОРРЕЛЯЦИИ МЕДИАПОКАЗАТЕЛЕЙ С ЧИСЛОМ ЗАЧИСЛЕННЫХ")
    print("="*80)
    
    # Загружаем и агрегируем данные
    print("\nЗагрузка файлов:")
    aggregated = load_and_aggregate_data()
    
    # Анализируем каждую метрику
    results = {}
    
    metrics = [
        ('posts', 'Количество постов'),
        ('likes', 'Сумма лайков'),
        ('reposts', 'Сумма репостов'),
        ('comments', 'Сумма комментариев'),
        ('avg_er', 'Средний ER Post')
    ]
    
    for metric, label in metrics:
        results[metric] = analyze_correlation(aggregated, metric, label)
    
    # Выводим таблицу
    print_analysis_table(results)
    
    # Строим графики
    plot_correlation(results, 'корреляция_зачисление.png')
    plot_forecast(results, 'прогноз_2026.png')
    
    print("\nАнализ завершен!")

if __name__ == "__main__":
    main()

АНАЛИЗ КОРРЕЛЯЦИИ МЕДИАПОКАЗАТЕЛЕЙ С ЧИСЛОМ ЗАЧИСЛЕННЫХ

Загрузка файлов:
  Загружен Profi-23.xlsx: 63 постов
  Загружен ZTTiE-23.xlsx: 568 постов

ИТОГИ 2023:
  Постов: 631
  Лайков: 35689
  Репостов: 3052
  Комментариев: 686
  Средний ER: 1.2811
  Зачислено: 686
  Загружен Profi-24.xlsx: 222 постов
  Загружен ZTTiE-24.xlsx: 729 постов

ИТОГИ 2024:
  Постов: 951
  Лайков: 106136
  Репостов: 3349
  Комментариев: 760
  Средний ER: 3.0902
  Зачислено: 768
  Загружен Profi-25.xlsx: 210 постов
  Загружен ZTTiE-25.xlsx: 635 постов

ИТОГИ 2025:
  Постов: 845
  Лайков: 167835
  Репостов: 3507
  Комментариев: 2019
  Средний ER: 6.3636
  Зачислено: 808

РЕЗУЛЬТАТЫ КОРРЕЛЯЦИОННОГО АНАЛИЗА
           Метрика Корреляция    R² Сила связи   Направление         Уравнение Прогноз 2026
 Количество постов      0.791 0.625    Сильная Положительная  y = 0.30·x + 510          797
      Сумма лайков      0.988 0.975    Сильная Положительная  y = 0.00·x + 658          875
    Сумма репостов      1.000 1.000 